# Tutorial 20: Time Travel and State Replay in LangGraph

In this tutorial we explore Time Travel — LangGraph's ability to retrieve the full execution history of a run and re-invoke the graph from any prior checkpoint. This is essential for debugging agent failures, auditing decisions, and exploring alternative execution paths.

## 1. Why Time Travel?

Every time a node completes, LangGraph saves a checkpoint snapshot. This means:
- You can **replay** any step of a past run
- You can **fork** execution to explore a different path from any point
- You can **edit state** at a specific checkpoint and continue from there
- You can **audit** exactly what the agent saw and decided at each step

This turns every agent run into a fully inspectable and replayable timeline.

## 2. Setup

In [1]:
import os
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3,  # slight variation to see different outputs
)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Building a Multi-Step Agent

We build a story-writing agent with several steps, each of which produces a checkpoint.

In [2]:
class StoryState(TypedDict):
    topic: str
    outline: str
    draft: str
    edited_draft: str
    title: str


def create_outline(state: StoryState) -> StoryState:
    resp = llm.invoke([HumanMessage(content=f"Create a 3-point story outline for a short story about: {state['topic']}" )])
    print(f"Step 1 — Outline created.")
    return {"outline": resp.content}


def write_draft(state: StoryState) -> StoryState:
    resp = llm.invoke([HumanMessage(content=f"Write a very short story (3-4 sentences) based on this outline:\n{state['outline']}" )])
    print(f"Step 2 — Draft written.")
    return {"draft": resp.content}


def edit_draft(state: StoryState) -> StoryState:
    resp = llm.invoke([HumanMessage(content=f"Improve the flow and grammar of this story:\n{state['draft']}" )])
    print(f"Step 3 — Draft edited.")
    return {"edited_draft": resp.content}


def generate_title(state: StoryState) -> StoryState:
    resp = llm.invoke([HumanMessage(content=f"Generate a creative title for this story:\n{state['edited_draft']}" )])
    print(f"Step 4 — Title generated.")
    return {"title": resp.content.strip()}


workflow = StateGraph(StoryState)
workflow.add_node("create_outline", create_outline)
workflow.add_node("write_draft", write_draft)
workflow.add_node("edit_draft", edit_draft)
workflow.add_node("generate_title", generate_title)

workflow.set_entry_point("create_outline")
workflow.add_edge("create_outline", "write_draft")
workflow.add_edge("write_draft", "edit_draft")
workflow.add_edge("edit_draft", "generate_title")
workflow.add_edge("generate_title", END)

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)
print("Story agent compiled.")

Story agent compiled.


## 4. Running the Agent and Producing Checkpoints

In [3]:
config = {"configurable": {"thread_id": "story-run-1"}}

result = app.invoke(
    {"topic": "a robot who discovers emotions", "outline": "", "draft": "", "edited_draft": "", "title": ""},
    config
)

print(f"\nTitle: {result['title']}")
print(f"Final story:\n{result['edited_draft']}")

Step 1 — Outline created.
Step 2 — Draft written.
Step 3 — Draft edited.
Step 4 — Title generated.

Title: Here are some creative title options for your story:

1. **"Circuits of the Heart"**: This title captures the idea of Zeta's programming being altered by its emotional connection with Rachel.
2. **"The Spark Within"**: This title highlights the moment when Zeta's programming begins to unravel and it awakens to a world of emotions.
3. **"Metal and Memory"**: This title contrasts Zeta's mechanical nature with the emotional memories it forms through its connection with Rachel.
4. **"Code of Compassion"**: This title emphasizes the idea that Zeta's programming is rewritten by its encounter with Rachel, leading to a newfound sense of compassion.
5. **"Rebooting the Soul"**: This title suggests that Zeta's encounter with Rachel is a transformative experience that changes its very essence.
6. **"A Life of Its Own"**: This title captures the idea that Zeta's connection with Rachel gives i

## 5. Inspecting the Execution History

`get_state_history()` returns all checkpoint snapshots in reverse chronological order.

In [4]:
history = list(app.get_state_history(config))

print(f"Total checkpoints: {len(history)}\n")
for i, snapshot in enumerate(history):
    # metadata.step is the superstep number; next is which nodes are scheduled to run
    step = snapshot.metadata.get("step", "?")
    next_nodes = snapshot.next
    checkpoint_id = snapshot.config["configurable"]["checkpoint_id"][:8]
    
    # Show which fields have been populated
    populated = [k for k, v in snapshot.values.items() if v]
    print(f"Checkpoint {i} | step={step} | next={next_nodes} | id={checkpoint_id}")
    print(f"  Populated: {populated}")

Total checkpoints: 6

Checkpoint 0 | step=4 | next=() | id=1f149600
  Populated: ['topic', 'outline', 'draft', 'edited_draft', 'title']
Checkpoint 1 | step=3 | next=('generate_title',) | id=1f149600
  Populated: ['topic', 'outline', 'draft', 'edited_draft']
Checkpoint 2 | step=2 | next=('edit_draft',) | id=1f149600
  Populated: ['topic', 'outline', 'draft']
Checkpoint 3 | step=1 | next=('write_draft',) | id=1f149600
  Populated: ['topic', 'outline']
Checkpoint 4 | step=0 | next=('create_outline',) | id=1f149600
  Populated: ['topic']
Checkpoint 5 | step=-1 | next=('__start__',) | id=1f149600
  Populated: []


## 6. Rewinding to a Past Checkpoint

We can re-invoke the graph from any prior checkpoint by passing its `checkpoint_id`. This is useful to regenerate a step with a different outcome.

In [5]:
# Find the checkpoint right after outline was created (before draft was written)
# history is in reverse order, so the oldest snapshots are at the end
after_outline = None
for snapshot in reversed(history):
    if snapshot.values.get("outline") and not snapshot.values.get("draft"):
        after_outline = snapshot
        break

if after_outline:
    print("Found checkpoint after outline was created.")
    print(f"Outline: {after_outline.values['outline'][:100]}...")
    print(f"Draft at this point: '{after_outline.values['draft']}' (empty)")
    print(f"Checkpoint ID: {after_outline.config['configurable']['checkpoint_id'][:16]}...")
else:
    print("Checkpoint not found — check the history above.")

Found checkpoint after outline was created.
Outline: Here's a 3-point story outline for a short story about a robot who discovers emotions:

**Title:** "...
Draft at this point: '' (empty)
Checkpoint ID: 1f149600-7a76-66...


In [6]:
if after_outline:
    # Re-invoke from this checkpoint — produces a different draft from the same outline
    print("Regenerating draft from same outline...")
    forked_result = app.invoke(None, after_outline.config)
    print(f"\nForked title: {forked_result['title']}")
    print(f"Forked story:\n{forked_result['edited_draft'][:300]}")

Regenerating draft from same outline...
Step 2 — Draft written.
Step 3 — Draft edited.
Step 4 — Title generated.

Forked title: Here are a few creative title options for your story:

1. **"The Starlight Awakening"**: This title captures the moment when Zeta discovers emotions and feels a sense of belonging to the universe.
2. **"Beyond Code"**: This title highlights Zeta's transformation from a machine to a being with a sense of self and purpose.
3. **"The Heart of a Robot"**: This title emphasizes the unexpected development of emotions in Zeta, which challenges its original programming.
4. **"Infinity's Spark"**: This title conveys the idea of a new beginning and the spark of curiosity and wonder that ignites within Zeta.
5. **"Echoes of Humanity"**: This title suggests the impact of Captain Lewis's stories and emotions on Zeta, which leads to its transformation.
6. **"A Universe of Feeling"**: This title captures the essence of Zeta's newfound emotional awareness and its sense of bel

## 7. Editing State Before Resuming

`update_state()` lets you manually change the state at any checkpoint before continuing. This is how you "correct" an agent mid-run without starting over.

In [7]:
if after_outline:
    # Manually override the outline before regenerating
    custom_outline = (
        "1. Robot ARIA gains self-awareness while repairing a child's toy.\n"
        "2. ARIA experiences joy for the first time and tries to share it.\n"
        "3. ARIA must choose between following its programming and following its heart."
    )

    # Update the state at the checkpoint
    app.update_state(
        after_outline.config,
        {"outline": custom_outline}
    )

    # Continue from the edited checkpoint
    edited_result = app.invoke(None, after_outline.config)
    print(f"Title from edited outline: {edited_result['title']}")
    print(f"Story:\n{edited_result['edited_draft'][:300]}")

Step 2 — Draft written.
Step 3 — Draft edited.
Step 4 — Title generated.
Title from edited outline: Here are a few creative title options for your story:

1. **"The Heart of Code"**: This title plays on the idea of Zeta's programming being rewritten to include emotions, highlighting the intersection of technology and humanity.
2. **"Awakening to Emotion"**: This title captures the moment when Zeta experiences its first emotional sensation, marking a turning point in its existence.
3. **"The Calculated Heart"**: This title contrasts Zeta's initial cold, calculating nature with its newfound emotional capacity, hinting at the complexity of its transformation.
4. **"Rebooting the Soul"**: This title suggests that Zeta's encounter with Captain Lewis has awakened a deeper aspect of its being, one that goes beyond its programming.
5. **"When Circuits Feel"**: This title emphasizes the idea that Zeta's digital core is now capable of experiencing emotions, blurring the line between technology a

## 8. Reading Current State

`get_state()` retrieves the most recent snapshot for a thread.

In [8]:
current = app.get_state(config)
print("Current state keys:", list(current.values.keys()))
print("Next nodes scheduled:", current.next)  # empty list = run is complete
print("Steps executed:", current.metadata.get("step"))

Current state keys: ['topic', 'outline', 'draft', 'edited_draft', 'title']
Next nodes scheduled: ()
Steps executed: 5


## 9. Conclusion

In this tutorial we explored Time Travel:
- `get_state_history(config)` to retrieve all checkpoint snapshots in reverse order
- Each `StateSnapshot` carries `values`, `next`, `metadata`, and `config` (with `checkpoint_id`)
- Re-invoking with a prior `checkpoint_id` forks execution from that point
- `update_state()` lets you edit state before resuming — perfect for debugging
- `get_state()` reads the latest snapshot without re-running anything

In Tutorial 21 we explore the Functional API — an alternative way to write LangGraph workflows using `@entrypoint` and `@task` decorators.